# SV-LT-RECH Stage 1 simulation study (Colab)

End-to-end runner for Phase 3 of the `finance_eng` project. Installs MATLAB on the Colab VM, activates a license, runs the Stage 1 parameter-recovery study, and bundles the outputs for download as the Phase 4 handoff archive.

**Reference:** [MathWorks blog: Using MATLAB on Google Colab](https://blogs.mathworks.com/matlab/2025/06/27/using-matlab-on-google-colab/)

## Order of operations
1. Upload / unpack the `finance_eng_colab.tar.gz` bundle.
2. Install MATLAB R2026a + Statistics Toolbox via `mpm`.
3. License-activate (one-time per VM session; needs MathWorks OTP).
4. Run the **quick** variant to verify the install + code path (~30 min).
5. Inspect the gate verdict; if green, launch the **full** study.
6. After completion, package the results and download the archive.
7. Ship the archive back to the assistant; Phase 4 ingests it.

## 1. Upload + unpack the bundle

Two options:

**A. Direct upload** -- drag `finance_eng_colab.tar.gz` into the Colab Files sidebar (`/content/`), then run the cell below.

**B. Drive sync** -- mount Drive and `cp` the bundle from there.

In [ ]:
import os, glob
%cd /content

bundles = glob.glob('finance_eng_colab*.tar.gz')
assert bundles, 'Upload finance_eng_colab.tar.gz to /content first.'
bundle = sorted(bundles)[-1]
print('Using bundle:', bundle)

!mkdir -p /content/finance_eng_colab && tar -xzf {bundle} -C /content/finance_eng_colab
%cd /content/finance_eng_colab
!ls -la

## 2. Install MATLAB R2026a + Statistics Toolbox

Takes 5-10 minutes the first time. Idempotent: re-running is fast if MATLAB is already installed.

In [ ]:
!bash install_matlab.sh

## 3. License-activate (one-time per Colab VM)

MATLAB R2026a uses online licensing. The activation flow is **interactive**, so it has to run in a Colab **terminal**, not as a notebook `!` cell.

1. Open a Colab terminal: **Tools → Terminal** (or click the `>_` icon at the bottom-left).
2. In that terminal, run:
   ```bash
   /usr/local/MATLAB/R2026a/bin/matlab -nodesktop -licmode onlinelicensing
   ```
3. When prompted, enter your MathWorks account email.
4. Visit <https://www.mathworks.com/mwa/otp> in another tab to get a one-time password.
5. Paste the OTP back in the terminal. MATLAB caches the license for the rest of the Colab VM session (~12 h).
6. Type `exit` to leave that MATLAB session. The notebook cells below now run with the cached license.

Reference: [MathWorks blog → *Using MATLAB on Google Colab*](https://blogs.mathworks.com/matlab/2025/06/27/using-matlab-on-google-colab/) — R2025a in the post, R2026a here.

In [ ]:
# Verify the cached online license is live before running the study.
# (You only need this AFTER completing the OTP flow in the Colab terminal above.)
# The -licmode onlinelicensing flag is REQUIRED on every batch launch -- without it
# MATLAB falls back to file-based licensing and errors out with -1.2.
!matlab -licmode onlinelicensing -batch "disp(version); disp('license OK')" 2>&1 | tail -20

## 4. QUICK sanity run (~30 min)

Runs `stage1_colab_quick.json`: 5 replicates x 2 seeds at small SMC scale. Verifies the install + code path. The gate verdict will likely be FAIL because the study is undersized -- that is fine; the purpose is to confirm nothing crashes.

In [ ]:
import os
os.environ['STAGE1_CONFIG'] = 'stage1_colab_quick.json'

!matlab -licmode onlinelicensing -batch "addpath('matlab_src'); run('matlab_src/scripts/run_stage1_colab.m')" 2>&1 | tee run_quick.log

In [ ]:
import json, pathlib
p = pathlib.Path('results/stage1_colab_quick/gate_summary.json')
if p.is_file():
    print(json.dumps(json.loads(p.read_text()), indent=2))
else:
    print('No gate_summary.json yet -- check run_quick.log for errors.')

!ls -la results/stage1_colab_quick/ 2>/dev/null || true

## 5. FULL study (multi-session, ~hours per Colab VM)

Runs `stage1_colab.json`: 30 replicates x 3 seeds at full SMC scale (N=2000, M=200, T=2000). The runner is **resumable**: if the Colab VM expires before completion, re-run the cell and it picks up from the last completed replicate.

Wall-clock estimate: many hours; potentially multiple Colab sessions. After each replicate finishes, partial CSVs + posterior `.mat` files are flushed to disk so progress is never lost.

In [ ]:
import os
os.environ['STAGE1_CONFIG'] = 'stage1_colab.json'

!matlab -licmode onlinelicensing -batch "addpath('matlab_src'); run('matlab_src/scripts/run_stage1_colab.m')" 2>&1 | tee -a run_full.log

In [ ]:
# Progress snapshot at any point.
!tail -30 run_full.log
print()
!wc -l results/stage1_colab/recovery.csv 2>/dev/null || echo 'no recovery.csv yet'
!ls results/stage1_colab/posteriors/ 2>/dev/null | wc -l && echo 'posterior .mat files'

## 6. Package results for Phase 4 handoff

In [ ]:
!matlab -licmode onlinelicensing -batch "run('matlab_src/scripts/package_results.m')"

In [ ]:
# Download the archive to your local machine.
import glob
from google.colab import files

archives = sorted(glob.glob('phase4_handoff_*.tar.gz'))
assert archives, 'package_results.m did not produce an archive.'
latest = archives[-1]
print('Downloading:', latest)
files.download(latest)

## What to ship back

Send the downloaded `phase4_handoff_*.tar.gz` archive back to the assistant. See `PHASE_4_HANDOFF.md` in this bundle for the exact contents the assistant expects.